# 🌾 AgriSense AI — Crop Recommendation Model Training
**Run all cells in order → Download the two .pkl files at the end**

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print('✅ All packages ready')

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports done')

In [ ]:
# ── Cell 3: Generate Synthetic Dataset ────────────────────────────────────────
N = 1200

def make_crop_samples(crop, n, temp_range, hum_range, rain_range, moist_range,
                       ph_range, n_range, p_range, k_range):
    return pd.DataFrame({
        'temperature':    np.random.uniform(*temp_range,  n),
        'humidity':       np.random.uniform(*hum_range,   n),
        'rainfall':       np.random.uniform(*rain_range,  n),
        'soil_moisture':  np.random.uniform(*moist_range, n),
        'soil_ph':        np.random.uniform(*ph_range,    n),
        'nitrogen':       np.random.uniform(*n_range,     n),
        'phosphorus':     np.random.uniform(*p_range,     n),
        'potassium':      np.random.uniform(*k_range,     n),
        'crop_label':     crop
    })

per_crop = N // 5
datasets = [
    make_crop_samples('rice',      per_crop, (22,32),(70,90),(1000,2500),(55,90),(5.5,6.5),(80,120),(40,80),(30,60)),
    make_crop_samples('wheat',     per_crop, (15,25),(50,70),(400,700), (30,60),(6.0,7.5),(60,100),(30,60),(20,50)),
    make_crop_samples('sugarcane', per_crop, (24,38),(70,90),(1500,2500),(60,90),(6.0,7.0),(100,150),(50,90),(50,90)),
    make_crop_samples('cotton',    per_crop, (20,35),(50,75),(700,1200),(35,65),(6.0,7.5),(60,100),(30,70),(30,70)),
    make_crop_samples('groundnut', per_crop, (25,35),(50,70),(500,800), (30,55),(5.5,7.0),(50,90),(40,80),(30,60)),
]
df = pd.concat(datasets, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(df['crop_label'].value_counts())
df.head()

In [ ]:
# ── Cell 4: Preprocess & Train ─────────────────────────────────────────────────
FEATURES = ['temperature','humidity','rainfall','soil_moisture','soil_ph','nitrogen','phosphorus','potassium']
TARGET   = 'crop_label'

X = df[FEATURES]
y = df[TARGET]

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n✅ Accuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── Cell 5: Confusion Matrix ───────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — AgriSense RF Model', fontsize=14)
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 6: Feature Importance ────────────────────────────────────────────────
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.plot(kind='barh', color='#43a047', figsize=(8,5))
plt.title('Feature Importances — AgriSense RF Model')
plt.xlabel('Importance')
plt.tight_layout(); plt.show()
print(fi.sort_values(ascending=False))

In [ ]:
# ── Cell 7: Save Models & Download ────────────────────────────────────────────
joblib.dump(rf, 'agrisense_rf_model.pkl')
joblib.dump(le, 'agrisense_label_encoder.pkl')
print('✅ Models saved!')

# Download in Colab
try:
    from google.colab import files
    print('Downloading agrisense_rf_model.pkl ...')
    files.download('agrisense_rf_model.pkl')
    print('Downloading agrisense_label_encoder.pkl ...')
    files.download('agrisense_label_encoder.pkl')
    print('✅ Both files downloaded! Place them in your agrisense_project/ folder.')
except Exception:
    print('Not in Colab — files saved locally as .pkl')